<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 1: Agentic AI Foundations

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Understand why agents exist** — see what LLMs can't do alone
2. **Learn what Agentic AI means** — the key difference from regular LLM calls
3. **Build a complete agent loop** — the core pattern behind all AI agents
4. **Define tools with JSON schemas** — teach an LLM to call your functions
5. **Explore the multi-model landscape** — use any LLM provider with the same code

---

## 1. Environment Setup

In [ ]:
!pip install -q openai rich

In [ ]:
import os
import json
from datetime import datetime
from getpass import getpass
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

---

## 2. The Problem — LLMs Can't Take Action

LLMs are incredibly smart — but they live inside a text box. They **can't** interact with the real world:

| What LLMs Can't Do | Example |
|---|---|
| **Access real-time data** | "What time is it?" — the LLM doesn't have a clock |
| **Run code or call APIs** | "Send an email to John" — it can only write the text |
| **Remember across sessions** | "What did I ask yesterday?" — no persistent memory |
| **Take actions in systems** | "Book a flight" — it can't click buttons |

Let's see this limitation firsthand. We'll ask the LLM a simple question it **should** be able to answer but can't:

In [ ]:
# Ask the LLM something it can't do — check the current time
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the exact current time right now?"}]
)

print("Question: What is the exact current time right now?")
print(f"\nLLM Response:\n{response.choices[0].message.content}")
print(f"\nActual time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

**Observation:** The LLM either guessed a time, gave a generic answer, or admitted it doesn't know. It **cannot** check a clock — it's just predicting text.

> The LLM is smart but trapped. It needs **hands** to interact with the world. That's what agents give it.

---

## 3. What is Agentic AI?

### The Chef vs. Cookbook Analogy

Think of the difference between a **cookbook** and a **chef**:

- A **cookbook** (regular LLM) gives you a recipe when asked — but it can't cook, taste, or adjust
- A **chef** (AI agent) reads the recipe, goes to the kitchen, cooks, tastes, adjusts seasoning, and serves the dish

An AI agent is an LLM that can **take actions** in a loop until a task is complete.

### Regular LLM vs. AI Agent

| | Regular LLM | AI Agent |
|---|---|---|
| **Input** | A question | A goal |
| **Output** | A text response | Actions + results |
| **Interaction** | One shot (ask → answer) | Loop (think → act → observe → repeat) |
| **Tools** | None | Functions, APIs, databases |
| **Example** | "Write an email" → gives you text | "Send an email" → actually sends it |
| **Analogy** | A cookbook | A chef |

### The Three Ingredients of an Agent

Every AI agent has three core ingredients:

```
┌─────────────────────────────────────────────┐
│              AI AGENT = LLM + TOOLS + LOOP  │
│                                             │
│  1. LLM (Brain)   → Decides what to do      │
│  2. Tools (Hands)  → Functions it can call   │
│  3. Loop (Process) → Keeps going until done  │
└─────────────────────────────────────────────┘
```

---

## 4. Simple Call vs. Agent — Side by Side

Let's see the difference in code. First, a **plain LLM call** — one question, one answer, done:

In [ ]:
# Plain LLM call — ask and get a text response
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is 2+2?"}]
)

print("Plain LLM call:")
print(f"  Answer: {response.choices[0].message.content}")
print(f"  Done? Yes — one shot, no follow-up")

That works fine for simple questions. But what about "What time is it right now?" The LLM can't do that alone — it needs **tools**.

The key insight: a plain LLM call is a **single step**. An agent is a **loop** that keeps calling tools until the job is done. Let's build one.

---

## 5. The Agent Loop

The agent loop is the core pattern behind **every** AI agent — from ChatGPT plugins to autonomous coding assistants. Here's how it works:

### The "Ordering Food" Analogy

Think of ordering food at a restaurant:

1. **You** tell the waiter what you want (user message)
2. **Waiter** (LLM) decides: go to the kitchen? Or respond directly?
3. If the waiter needs the kitchen (tool call), they go, get the food, and come back
4. **Repeat** until your order is complete
5. **Waiter** gives you the final response

### The Agent Loop — Step by Step

```
┌──────────────────────────────────────────────────────────┐
│                     THE AGENT LOOP                       │
│                                                          │
│  User Message                                            │
│       │                                                  │
│       ▼                                                  │
│  ┌─────────┐     tool_calls     ┌──────────┐            │
│  │   LLM   │──────────────────>│  Execute  │            │
│  │  (Brain) │                   │   Tools   │            │
│  │         │<──────────────────│  (Hands)  │            │
│  └─────────┘    tool results    └──────────┘            │
│       │                              ▲                   │
│       │ stop                         │                   │
│       ▼                     Loop until done              │
│  Final Response                                          │
└──────────────────────────────────────────────────────────┘
```

### How Does the LLM Decide?

The LLM returns a `finish_reason` with every response:

| `finish_reason` | Meaning | What We Do |
|---|---|---|
| `"tool_calls"` | "I need to use a tool" | Execute the tool, send results back |
| `"stop"` | "I'm done, here's my answer" | Return the response to the user |

That's the entire loop: **call LLM → check finish_reason → if tool_calls, execute and loop → if stop, return answer.**

---

## 6. Building Our First Agent — The Tool

Let's build an agent with a simple tool that gives it a capability the LLM doesn't have on its own:

1. **Tell the time** — something the LLM can't do alone

> **Note:** In the exercise at the end, you'll add a second tool yourself to practice the pattern.

In [ ]:
def get_current_time() -> dict:
    """Get the current date and time."""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"  [Tool] get_current_time → {now}")
    return {"time": now}

In [ ]:
# Quick test — make sure our tool works
print("Testing tool:\n")
get_current_time()

---

## 7. Tool Schemas — The "Restaurant Menu" for the LLM

Our Python function exists, but the LLM doesn't know about it yet. We need to describe the tool in a **JSON schema** — think of it as a **restaurant menu** for the LLM:

- The **menu** (schema) tells the customer (LLM) what dishes (tools) are available
- Each item has a **name**, **description**, and **ingredients** (parameters)
- The customer **orders** by name and specifies any customizations (arguments)

### Anatomy of a Tool Schema

```
{
    "name": "get_current_time",          ← Tool name (must match the function name)
    "description": "Get the current...", ← When should the LLM use this tool?
    "parameters": {
        "type": "object",
        "properties": {                  ← What inputs does the tool need?
            "param1": {
                "type": "string",        ← Data type
                "description": "..."     ← What this parameter is for
            }
        },
        "required": ["param1"],          ← Which parameters are mandatory?
        "additionalProperties": false    ← Don't allow extra parameters
    }
}
```

Let's define the schema for our tool:

In [ ]:
get_current_time_schema = {
    "name": "get_current_time",
    "description": "Get the current date and time",
    "parameters": {
        "type": "object",
        "properties": {},
        "additionalProperties": False
    }
}

# Package tools in the format the API expects
tools = [
    {"type": "function", "function": get_current_time_schema}
]

print(f"Defined {len(tools)} tool: {[t['function']['name'] for t in tools]}")

---

## 8. The Agent Loop — Code

Now we write the two functions that make our agent work:

1. **`handle_tool_calls`** — executes whatever tools the LLM requests
2. **`chat_with_tools`** — the main loop that keeps calling the LLM until it's done

In [ ]:
def handle_tool_calls(tool_calls):
    """Execute tool calls and return results to the LLM."""
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        print(f"  [Under the hood] The LLM returned:")
        print(f"    tool_call_id: {tool_call.id}")
        print(f"    function:     {tool_name}")
        print(f"    arguments:    {tool_call.function.arguments}")

        # Dynamic dispatch — look up the function by name
        tool_function = globals().get(tool_name)

        if tool_function:
            result = tool_function(**arguments)
        else:
            result = {"error": f"Unknown tool: {tool_name}"}

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

In [ ]:
system_prompt = """You are a helpful assistant. You can tell the user the current time using the get_current_time tool. For all other questions, answer using your own knowledge."""

def chat_with_tools(message, history=[]):
    """The main agent loop — keeps calling the LLM until it stops."""
    messages = [
        {"role": "system", "content": system_prompt}
    ] + history + [
        {"role": "user", "content": message}
    ]

    done = False
    while not done:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        finish_reason = response.choices[0].finish_reason
        print(f"  [Loop] finish_reason = {finish_reason}")

        if finish_reason == "tool_calls":
            # LLM wants to use tools — execute them and loop
            assistant_message = response.choices[0].message
            tool_results = handle_tool_calls(assistant_message.tool_calls)
            messages.append(assistant_message)
            messages.extend(tool_results)
        else:
            # LLM is done — return the final response
            done = True

    return response.choices[0].message.content

---

## 9. Run It!

Let's test our agent with a question that requires the tool, then one that doesn't:

In [ ]:
# Scenario 1: Ask for the time (uses get_current_time tool)
print("=" * 50)
print("Scenario 1: What time is it?")
print("=" * 50)
response = chat_with_tools("What time is it?")
print(f"\nAgent: {response}")

In [ ]:
# Scenario 2: No tool needed (LLM answers directly — finish_reason = "stop")
print("=" * 50)
print("Scenario 2: No tool needed")
print("=" * 50)
response = chat_with_tools("What is the capital of France?")
print(f"\nAgent: {response}")

### What Just Happened?

| Scenario | User Said | Tool Called | Why |
|---|---|---|---|
| 1. Time | "What time is it?" | `get_current_time` | LLM can't check a clock — needs the tool |
| 2. Capital | "What is the capital of France?" | *None* | LLM already knows the answer — responds directly |

Notice two things:
1. The LLM **decided on its own** which tool to call based on the conversation. We never told it "if the user asks about time, call `get_current_time`." It figured that out from the tool description in our schema.
2. When no tool was needed, the LLM took the **stop** path — it just answered directly without calling any tool.

---

## 10. The Multi-Model Landscape

So far we've used OpenAI's `gpt-4o-mini`. But the AI world has **many** LLM providers — and a key insight is that most of them use the **same API format** as OpenAI. This means you can switch models by just changing the URL and API key!

### Provider Comparison

| Provider | Model | API Style | Cost | Best For |
|---|---|---|---|---|
| **OpenAI** | GPT-4o, GPT-4o-mini | Native | Paid | General purpose, tool calling |
| **Anthropic** | Claude 4, Claude Haiku | Own SDK | Paid | Long context, safety, reasoning |
| **Google** | Gemini 1.5, 2.0 | OpenAI-compatible | Paid/Free tier | Multimodal, large context |
| **DeepSeek** | DeepSeek-V3, R1 | OpenAI-compatible | Very cheap | Reasoning, coding |
| **Groq** | Llama, Mixtral | OpenAI-compatible | Free tier | Ultra-fast inference |
| **Ollama** | Llama, Mistral, Phi | OpenAI-compatible | Free (local) | Privacy, offline use |

### The OpenAI-Compatible Pattern

Most providers support the same `OpenAI` client — you just change `base_url` and `api_key`:

```python
# OpenAI (default)
client = OpenAI(api_key="sk-...")

# Google Gemini (same client, different URL)
client = OpenAI(api_key="your-google-key", base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# Groq (same client, different URL)
client = OpenAI(api_key="your-groq-key", base_url="https://api.groq.com/openai/v1")

# Ollama (local, no API key needed)
client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
```

This is powerful — you write your agent code **once** and swap models freely.

In [ ]:
# Demo: Use the same question with a different provider (Groq — free and fast)
# If you don't have a Groq key, skip this cell — it's just a demo

groq_key = getpass("Enter Groq API Key (free at console.groq.com, or press Enter to skip): ")

if groq_key:
    groq_client = OpenAI(
        api_key=groq_key,
        base_url="https://api.groq.com/openai/v1"
    )

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "What is Agentic AI in one sentence?"}]
    )
    print(f"Groq (Llama 3.3 70B):\n{response.choices[0].message.content}")
else:
    print("Skipped — no Groq API key provided")
    print("You can get a free key at console.groq.com")

---

## 11. Concept Map

Here's how everything we learned connects:

```
┌──────────────────────────────────────────────────────────────────┐
│                     AGENTIC AI — BIG PICTURE                    │
│                                                                  │
│  ┌────────────┐                                                  │
│  │  TOOLS     │ Python functions + JSON schemas                  │
│  │  (Hands)   │ get_current_time (+ any tools you define)        │
│  └─────┬──────┘                                                  │
│        │                                                         │
│        ▼                                                         │
│  ┌────────────┐     ┌───────────┐                                │
│  │  LLM       │────>│  DECIDES  │──> call a tool? or respond?    │
│  │  (Brain)   │     └───────────┘                                │
│  └─────┬──────┘                                                  │
│        │                                                         │
│        ▼                                                         │
│  ┌────────────┐                                                  │
│  │  LOOP      │ Repeats until finish_reason = "stop"             │
│  │  (Process) │                                                  │
│  └────────────┘                                                  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  PROVIDERS: OpenAI │ Anthropic │ Google │ Groq │ Ollama   │  │
│  │  Same pattern, different base_url + api_key               │  │
│  └────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────┘
```

---

## 12. Key Takeaways

1. **LLMs are smart but trapped** — they can't access real-time data, call APIs, or take actions without tools

2. **Agentic AI = LLM + Tools + Loop** — give the LLM functions to call and a loop to keep working

3. **Tool schemas are the menu** — JSON descriptions that tell the LLM what tools exist and how to use them

4. **The agent loop is simple** — call LLM, check `finish_reason`, execute tools if needed, repeat until done

5. **Most LLM providers are OpenAI-compatible** — write your code once, swap models by changing `base_url`

### Stage-to-Code Mapping

| Concept | Our Code | What It Does |
|---|---|---|
| **Tool** | `get_current_time()` | Python function the agent can call |
| **Schema** | `get_current_time_schema` | JSON description of the tool |
| **Tool Executor** | `handle_tool_calls()` | Runs the tool and returns results |
| **Agent Loop** | `chat_with_tools()` | Loops until the LLM says "stop" |
| **Multi-Model** | `OpenAI(base_url=...)` | Same code, different provider |

---

## 13. Exercise: Add a `get_weather` Tool

Add a tool that returns the weather for a city. Since this is practice, just return hardcoded/fake data — the goal is to practice the pattern.

1. Write the function: `get_weather(city)` → returns `{"city": city, "temperature": "28°C", "condition": "Sunny"}`
2. Write the JSON schema (one required parameter: `city`)
3. Add it to the `tools` list
4. Test: `chat_with_tools("What's the weather in Bhubaneswar?")`

In [ ]:
# Your code here

---

**What's Next?** In the next notebook, we'll make our agents **reliable** — with structured outputs, quality evaluation, retry loops, and a complete capstone project.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---